In [2]:
import os  # Used for file management (removing files, path operations)
import pandas as pd  # Used for handling tabular data (reading/writing CSV files)
import subprocess  # Used to run hmm software as an external process
from tqdm import tqdm  # Used to display a progress bar for tracking processing
from os import listdir  # Used to list files in a directory
from Bio import SeqIO  # Used to read and parse FASTA files
from Bio.Seq import Seq  # Used for reverse complementing sequences

import json

**HMM database**: The pfam_path points to a .hmm file containing hmm profiles for many domains, including those in the domains list. This file is 'pressed' to optimize it for storage.

**Scanning**: For each gene sequence, run HMMERs hmmscan which;
    - Aligns the sequence against all HMM profiles in the database
    - returns hits (matching domains), scores, biases, and ranges

**Scoring**: Filter the results to only record scores for domains in the domains list.
    - If a domain from domains list is detected, its score is stored, otherwise, it remains 0.
    - The result is a feature matrix where rows are genes and columns are domain scores.

**Feature Integration**: These scores are concatenated with protein embeddings and fed into the XGBoost model to predict RBPs. The HMM score indicate the presence and strenth of RBP-related domains, enhancing the model's ability to classify proteins.

In [21]:
dir_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/'
phage_file = 'data/phage_genomes/A1a.fasta'

xgb_path = 'RBPdetect_xgb_hmm.json'
phage_path = os.path.abspath(os.path.join(dir_path, phage_file))

genebase = pd.read_csv('data/phanotate/genebase_embeddings.csv')

In [20]:
# to load json pfam domains
with open('rbp_domains.json', 'r') as f:
    data = json.load(f)

domains = data['new_blocks']
print(domains)


['Phage_T7_tail', 'Tail_spike_N', 'Prophage_tail', 'BppU_N', 'Mtd_N', 'Head_binding', 'DUF3751', 'End_N_terminal', 'phage_tail_N', 'Prophage_tailD1', 'DUF2163', 'Phage_fiber_2', 'unknown_N0', 'unknown_N1', 'unknown_N2', 'unknown_N3', 'unknown_N4', 'unknown_N6', 'unknown_N10', 'unknown_N11', 'unknown_N12', 'unknown_N13', 'unknown_N17', 'unknown_N19', 'unknown_N23', 'unknown_N24', 'unknown_N26', 'unknown_N29', 'unknown_N36', 'unknown_N45', 'unknown_N48', 'unknown_N49', 'unknown_N53', 'unknown_N57', 'unknown_N60', 'unknown_N61', 'unknown_N65', 'unknown_N73', 'unknown_N82', 'unknown_N83', 'unknown_N101', 'unknown_N114', 'unknown_N119', 'unknown_N122', 'unknown_N163', 'unknown_N174', 'unknown_N192', 'unknown_N200', 'unknown_N206', 'unknown_N208', 'Lipase_GDSL_2', 'Pectate_lyase_3', 'gp37_C', 'Beta_helix', 'Gp58', 'End_beta_propel', 'End_tail_spike', 'End_beta_barrel', 'PhageP22-tail', 'Phage_spike_2', 'gp12-short_mid', 'Collar', 'unknown_C2', 'unknown_C3', 'unknown_C8', 'unknown_C15', 'unkn

In [17]:
def hmmpress(hmm_path, pfam_file):
    """
    Prepares an HMM profiles database (pfam_file) for efficient querying by HMMERs hmmscan.
    - This is a one-time setup.
    - hmmpress compressses and indexes the .hmm file, generating auxilary files (.h3m, .h3i) needed for fast searches.

    Inputs:
        hmm_path: path to HMMER software
        pfam_file: Path to the HMM profiles 
    """
    cd_str = "cd " + hmm_path 
    press_str = 'hmmpress ' + pfam_file
    command = cd_str + ';' + press_str
    press_process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    stdout, _ = press_process.communicate()
    if press_process.returncode != 0:
        raise RuntimeError(f"Error running hmmpress: {stdout.decode()} ")

In [ ]:
hmm_path = '/home/dylan33smith/src/hmmer-3.4/src'
pfam_path = '/home/dylan33smith/projects/Yuzhen/PB_interactions/RBPdetect_phageRBPs.hmm'

hmmpress(hmm_path, pfam_path)